## Частина 2: Аналіз датасету Individual Household Electric Power Consumption

### Завдання 1: Завантаження даних та Data Cleaning
**Умова:** Завантажити, зчитати датасет та здійснити очищення даних (обробка пропущених значень `?`, видалення рядочків із NaN, приведення числових атрибутів до належних типів). Провести профілювання часу виконання процедури за допомогою модуля `timeit`.

In [12]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import timeit

os.makedirs("../datasets", exist_ok=True)
zip_path = "../datasets/household_power_consumption.zip"
file_path = "../datasets/household_power_consumption.txt"
dataset_url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"

if not os.path.exists(file_path):
    print("--- Файл датасету не знайдено! Починаємо автоматичне завантаження з UCI... ---")
    try:
        urllib.request.urlretrieve(dataset_url, zip_path)
        print("Архів успішно завантажено. Розпакування...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("../datasets")
        os.remove(zip_path)
        print("--- Датасет успішно розпаковано і готово до роботи! ---\n")
    except Exception as e:
        print(f"Помилка під час завантаження або розпакування: {e}")
else:
    print("--- Датасет уже присутній локально, завантаження не потрібне. ---\n")


def load_and_clean_power_data(path):
    # 'na_values=?' автоматично замінить знаки питання на NaN при зчитуванні
    df = pd.read_csv(path, sep=';', na_values='?', low_memory=False)
    
    # Видаляємо всі рядки, які містять пропущені значення
    df = df.dropna()
    
    # Стовпчики, які необхідно явно перетворити на float
    numeric_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage', 
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    for col in numeric_cols:
        df[col] = df[col].astype(float)
        
    # Створюємо єдиний об'єкт Datetime для зручної фільтрації за часом
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
    return df

# Вимірюємо час виконання процедури (1 прогін)
execution_time = timeit.timeit(lambda: load_and_clean_power_data(file_path), number=1)

# Зберігаємо очищений датафрейм у змінну для подальшої роботи
power_df = load_and_clean_power_data(file_path)

print("--- Дані УСПІШНО завантажено та очищено! ---")
print(f"Час виконання процедури завантаження: {execution_time:.4f} секунд")
print(f"Розмірність очищеного датасету (рядків, колонок): {power_df.shape}")
print("\nПерші 5 рядків очищеного датасету:")
display(power_df.head())

--- Датасет уже присутній локально, завантаження не потрібне. ---

--- Дані УСПІШНО завантажено та очищено! ---
Час виконання процедури завантаження: 4.9032 секунд
Розмірність очищеного датасету (рядків, колонок): (2049280, 10)

Перші 5 рядків очищеного датасету:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


### Завдання 2: Вибірка за активною потужністю (> 5 кВт)
**Умова:** Обрати всі записи, у яких загальна активна споживана потужність (`Global_active_power`) перевищує 5 кВт. Здійснити часове профілювання.

In [3]:
def filter_high_active_power(df):
    return df[df['Global_active_power'] > 5.0]

# Профілюємо середній час на основі 10 повторень
time_task1 = timeit.timeit(lambda: filter_high_active_power(power_df), number=10) / 10
res_task1 = filter_high_active_power(power_df)

print(f"Кількість знайдених записів: {res_task1.shape[0]}")
print(f"Середній час виконання фільтрації: {time_task1:.6f} секунд")
print("\nРезультат вибірки (перші 5 рядків):")
display(res_task1.head())

Кількість знайдених записів: 17547
Середній час виконання фільтрації: 0.006359 секунд

Результат вибірки (перші 5 рядків):


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00


### Завдання 3: Фільтрація за силою струму та порівняння груп споживання
**Умова:** Обрати всі записи, у яких сила струму (`Global_intensity`) лежить в межах 19-20 А. Серед них відібрати ті, у яких пральна машина, сушарка та освітлення (`Sub_metering_2`) споживають більше, ніж бойлер та кондиціонер (`Sub_metering_3`). Здійснити часове профілювання.

In [4]:
def filter_intensity_and_appliances(df):
    # Накладаємо комплексні умови через логічне "І" (&)
    condition = (df['Global_intensity'] >= 19.0) & \
                (df['Global_intensity'] <= 20.0) & \
                (df['Sub_metering_2'] > df['Sub_metering_3'])
    return df[condition]

# Профілюємо час (середнє за 10 запусків)
time_task2 = timeit.timeit(lambda: filter_intensity_and_appliances(power_df), number=10) / 10
res_task2 = filter_intensity_and_appliances(power_df)

print(f"Кількість знайдених записів: {res_task2.shape[0]}")
print(f"Середній час виконання фільтрації: {time_task2:.6f} секунд")
print("\nРезультат вибірки (перші 5 рядків):")
display(res_task2.head())

Кількість знайдених записів: 2509
Середній час виконання фільтрації: 0.008540 секунд

Результат вибірки (перші 5 рядків):


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


### Завдання 4: Випадкова вибірка та обчислення середніх значень груп споживання
**Умова:** Обрати випадковим чином 500,000 записів (без повторів елементів вибірки). Для отриманої вибірки обчислити середні величини усіх 3-х груп споживання електричної енергії (`Sub_metering_1`, `Sub_metering_2`, `Sub_metering_3`). Здійснити часове профілювання.

In [5]:
def get_random_sample_and_means(df, n=500000):
    # .sample(n=n, replace=False) гарантує випадковий вибір без повторів
    sample_df = df.sample(n=n, replace=False)
    
    # Обчислюємо середнє для трьох груп
    means = {
        'Sub_metering_1_mean': sample_df['Sub_metering_1'].mean(),
        'Sub_metering_2_mean': sample_df['Sub_metering_2'].mean(),
        'Sub_metering_3_mean': sample_df['Sub_metering_3'].mean()
    }
    return means

# Профілюємо час (середнє за 5 прогонів, бо вибірка велика)
time_task3 = timeit.timeit(lambda: get_random_sample_and_means(power_df), number=5) / 5
means_result = get_random_sample_and_means(power_df)

print(f"Середній час виконання випадкової вибірки та розрахунків: {time_task3:.6f} секунд")
print("\nСередні значення груп споживання для 500,000 випадкових записів:")
print(f"1. Кухня (Sub_metering_1): {means_result['Sub_metering_1_mean']:.4f} вт-год")
print(f"2. Пральня/Освітлення (Sub_metering_2): {means_result['Sub_metering_2_mean']:.4f} вт-год")
print(f"3. Бойлер/Кондиціонер (Sub_metering_3): {means_result['Sub_metering_3_mean']:.4f} вт-год")

Середній час виконання випадкової вибірки та розрахунків: 0.142549 секунд

Середні значення груп споживання для 500,000 випадкових записів:
1. Кухня (Sub_metering_1): 1.1174 вт-год
2. Пральня/Освітлення (Sub_metering_2): 1.2957 вт-год
3. Бойлер/Кондиціонер (Sub_metering_3): 6.4605 вт-год


### Завдання 5: Комплексна вечірня фільтрація та крокові зрізи
**Умова:** 1. Обрати записи після 18:00, де загальна активна потужність (`Global_active_power`) перевищує 6 кВт.
2. Серед них виділити ті, де група `Sub_metering_2` є строго більшою за `Sub_metering_1` та `Sub_metering_2` строго більша за `Sub_metering_3`.
3. Розділити отриманий датафрейм навпіл.
4. З першої половини обрати кожен 3-й результат, а з другої половини — кожен 4-й результат.
5. Здійснити часове профілювання.

In [7]:
def complex_evening_filter(df):
    # Фільтруємо за часом (після 18:00) та потужністю (> 6 кВт)
    # .dt.time дозволяє порівнювати чистий час без прив'зки до конкретної дати
    time_condition = df['Datetime'].dt.time > pd.to_datetime('18:00:00').time()
    power_condition = df['Global_active_power'] > 6.0
    
    initial_filter = df[time_condition & power_condition]
    
    # Фільтруємо, де група 2 є найбільшою серед усіх трьох
    group2_largest = initial_filter[
        (initial_filter['Sub_metering_2'] > initial_filter['Sub_metering_1']) & 
        (initial_filter['Sub_metering_2'] > initial_filter['Sub_metering_3'])
    ]
    
    if group2_largest.empty:
        return pd.DataFrame()
        
    # Знаходимо індекс середини датафрейму для розділення навпіл
    mid_index = len(group2_largest) // 2
    
    half_1 = group2_largest.iloc[:mid_index]
    half_2 = group2_largest.iloc[mid_index:]
    
    # Крокові зрізи: [::3] — кожен третій, [::4] — кожен четвертий
    sliced_1 = half_1.iloc[::3]
    sliced_2 = half_2.iloc[::4]
    
    # Об'єднуємо результати назад в один датафрейм
    final_result = pd.concat([sliced_1, sliced_2])
    return final_result

# Профілюємо час (середнє за 10 запусків)
time_task4 = timeit.timeit(lambda: complex_evening_filter(power_df), number=10) / 10
res_task4 = complex_evening_filter(power_df)

print(f"Кількість записів після фінальних зрізів: {res_task4.shape[0]}")
print(f"Середній час виконання комплексної процедури: {time_task4:.6f} секунд")
print("\nРезультат вибірки (перші 10 рядків):")
display(res_task4.head(10))

Кількість записів після фінальних зрізів: 310
Середній час виконання комплексної процедури: 0.557108 секунд

Результат вибірки (перші 10 рядків):


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00
17504,28/12/2006,21:08:00,7.352,0.000,235.45,31.2,1.0,73.0,17.0,2006-12-28 21:08:00
17507,28/12/2006,21:11:00,9.048,0.000,231.48,39.0,34.0,71.0,16.0,2006-12-28 21:11:00
17510,28/12/2006,21:14:00,9.118,0.108,231.18,39.4,36.0,72.0,16.0,2006-12-28 21:14:00
17513,28/12/2006,21:17:00,7.040,0.130,233.27,30.2,37.0,38.0,17.0,2006-12-28 21:17:00
18952,29/12/2006,21:16:00,6.146,0.116,230.53,26.6,0.0,70.0,0.0,2006-12-29 21:16:00


### Завдання 6: Нормування та стандартизація датасету
**Умова:** Провести нормування (MinMax Scaling в межах [0, 1]) та стандартизацію (Standard Scaling з математичним сподіванням 0 та дисперсією 1) для числових атрибутів вибраного датасету. Здійснити часове профілювання.

In [8]:
def scale_and_standardize(df):
    # Вибираємо лише числові колонки для трансформації
    cols_to_scale = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']
    
    # Створюємо копію, щоб не псувати оригінальний датафрейм
    scaled_df = df.copy()
    
    for col in cols_to_scale:
        # 1. Нормування (Min-Max Scaling)
        min_val = scaled_df[col].min()
        max_val = scaled_df[col].max()
        scaled_df[f'{col}_normalized'] = (scaled_df[col] - min_val) / (max_val - min_val)
        
        # 2. Стандартизація (Standard Scaling / Z-score)
        mean_val = scaled_df[col].mean()
        std_val = scaled_df[col].std()
        scaled_df[f'{col}_standardized'] = (scaled_df[col] - mean_val) / std_val
        
    return scaled_df

# Профілюємо час (середнє за 5 запусків)
time_task5 = timeit.timeit(lambda: scale_and_standardize(power_df), number=5) / 5
res_scaled = scale_and_standardize(power_df)

print(f"Середній час виконання нормування та стандартизації: {time_task5:.6f} секунд")
print("\nРезультат трансформації (оригінал vs нормований vs стандартизований для Global_active_power):")
display(res_scaled[['Global_active_power', 'Global_active_power_normalized', 'Global_active_power_standardized']].head())

Середній час виконання нормування та стандартизації: 0.762065 секунд

Результат трансформації (оригінал vs нормований vs стандартизований для Global_active_power):


,Global_active_power,Global_active_power_normalized,Global_active_power_standardized
0,4.216,0.374796,2.955076
1,5.360,0.478363,4.037084
2,5.374,0.479631,4.050325
3,5.388,0.480898,4.063566
4,3.666,0.325005,2.434881


### Завдання 7: Розрахунок коефіцієнтів кореляції Пірсона та Спірмена
**Умова:** Підрахувати коефіцієнти лінійної кореляції Пірсона та рангової кореляції Спірмена для двох числових атрибутів (`Global_active_power` та `Global_intensity`). Здійснити часове профілювання.

In [9]:
def calculate_correlations(df):
    # Розрахунок коефіцієнтів вбудованими методами pandas
    pearson_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
    spearman_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')
    return pearson_corr, spearman_corr

# Профілюємо час (1 прогін, бо рангова кореляція Спірмена на 2 млн рядків рахується довго через сортування)
time_task6 = timeit.timeit(lambda: calculate_correlations(power_df), number=1)

p_corr, s_corr = calculate_correlations(power_df)

print(f"Час розрахунку коефіцієнтів кореляції: {time_task6:.4f} секунд")
print(f"\nКоефіцієнт кореляції Пірсона: {p_corr:.6f}")
print(f"Коефіцієнт кореляції Спірмена: {s_corr:.6f}")

Час розрахунку коефіцієнтів кореляції: 3.0234 секунд

Коефіцієнт кореляції Пірсона: 0.998889
Коефіцієнт кореляції Спірмена: 0.995372


### Завдання 8: One Hot Encoding категоріального атрибута
**Умова:** Створити категоріальний атрибут на основі часу доби (`Time_Of_Day`) та провести його One Hot Encoding (бінарне кодування) за допомогою `pd.get_dummies()`. Здійснити часове профілювання.

In [15]:
# Попередньо додамо категоріальний атрибут «Час доби» до нашого датафрейму
def get_time_of_day(hour):
    if 5 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 22: return 'Evening'
    else: return 'Night'

# Створюємо категоріальну колонку
power_df['Time_Of_Day'] = power_df['Datetime'].dt.hour.apply(get_time_of_day)

def perform_one_hot_encoding(df):
    # Виконуємо One Hot Encoding для створеного стовпчика
    # columns=['Time_Of_Day'] вказує, яку саме колонку кодувати
    # dtype=int робить вихідні колонки 0 або 1 замість True/False
    return pd.get_dummies(df, columns=['Time_Of_Day'], dtype=int)

# Профілюємо час
time_task7 = timeit.timeit(lambda: perform_one_hot_encoding(power_df), number=5) / 5
res_encoded = perform_one_hot_encoding(power_df)

print(f"Середній час виконання One Hot Encoding: {time_task7:.6f} секунд")
print("\nРезультат кодування (нові бінарні колонки в кінці датафрейму):")
encoded_cols = [col for col in res_encoded.columns if 'Time_Of_Day' in col]
display(res_encoded[['Date', 'Time'] + encoded_cols].sample(10))

Середній час виконання One Hot Encoding: 0.136567 секунд

Результат кодування (нові бінарні колонки в кінці датафрейму):


,Date,Time,Time_Of_Day_Afternoon,Time_Of_Day_Evening,Time_Of_Day_Morning,Time_Of_Day_Night
2062972,18/11/2010,08:16:00,0,0,1,0
810602,1/7/2008,15:26:00,1,0,0,0
1667218,16/2/2010,12:22:00,1,0,0,0
774065,6/6/2008,06:29:00,0,0,1,0
1434458,7/9/2009,21:02:00,0,1,0,0
311616,21/7/2007,03:00:00,0,0,0,1
178922,19/4/2007,23:26:00,0,0,0,1
32288,8/1/2007,03:32:00,0,0,0,1
174522,16/4/2007,22:06:00,0,0,0,1
1285685,27/5/2009,13:29:00,1,0,0,0
